In [1]:
import pandas as pd
import json
from pathlib import Path
from PIL import Image
import shutil
import random
from tqdm.auto import tqdm

random.seed(67)

In [2]:
# Load class names and prompts
with open('data/plantwild/classes.txt', 'r') as f:
    classes = [line.strip().split(' ', 1)[1] for line in f.readlines()]

with open('data/plantwild/plantwild_prompts.json', 'r') as f:
    prompts_dict = json.load(f)

print(f"Total classes: {len(classes)}")

Total classes: 89


In [3]:
# Categorize classes into diseased and healthy
healthy_keywords = ['leaf', 'leaves']
diseased_classes = []
healthy_classes = []

for class_name in classes:
    # Check if class is healthy (ends with 'leaf' or 'leaves' and doesn't have disease keywords)
    words = class_name.split()
    if len(words) == 2 and words[-1] in healthy_keywords:
        healthy_classes.append(class_name)
    else:
        diseased_classes.append(class_name)

print(f"Diseased classes: {len(diseased_classes)}")
print(f"Healthy classes: {len(healthy_classes)}")
print(f"\nHealthy classes:")
for hc in healthy_classes:
    print(f"  - {hc}")

Diseased classes: 60
Healthy classes: 29

Healthy classes:
  - apple leaf
  - banana leaf
  - basil leaf
  - bean leaf
  - blueberry leaf
  - broccoli leaf
  - cabbage leaf
  - cauliflower leaf
  - celery leaf
  - cherry leaf
  - coffee leaf
  - corn leaf
  - cucumber leaf
  - eggplant leaf
  - garlic leaf
  - ginger leaf
  - grape leaf
  - lettuce leaf
  - maple leaf
  - peach leaf
  - plum leaf
  - potato leaf
  - raspberry leaf
  - rice leaf
  - soybean leaf
  - squash leaf
  - strawberry leaf
  - tobacco leaf
  - tomato leaf


In [4]:
# Create output directories
eval_base_dir = Path('data/plantwild/evaluation')
eval_images_dir = eval_base_dir / 'images'
eval_images_dir.mkdir(parents=True, exist_ok=True)

print(f"Created evaluation directory: {eval_base_dir}")

Created evaluation directory: data\plantwild\evaluation


In [5]:
# Sample evaluation images and MOVE them to evaluation folder
base_dir = Path('data/plantwild/images')
metadata = {}
image_counter = 0

# Use a fixed seed for reproducibility
random.seed(42)

In [6]:
# Sample diseased classes (3 images each) and MOVE them
print("Sampling diseased classes...")
diseased_count = 0

for class_name in tqdm(diseased_classes, desc="Diseased classes"):
    class_dir = base_dir / class_name
    
    if not class_dir.exists():
        print(f"Warning: Directory not found for {class_name}")
        continue
    
    # Get all image files
    image_files = list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.png')) + list(class_dir.glob('*.jpeg'))
    
    if len(image_files) < 3:
        print(f"Warning: Only {len(image_files)} images available for {class_name}")
    
    # Sample 3 images (or all if less than 3)
    sampled_files = random.sample(image_files, min(3, len(image_files)))
    
    # Get prompts for this class
    class_prompts = prompts_dict.get(class_name, [])
    
    for i, img_path in enumerate(sampled_files):
        try:
            # Create unique filename
            ext = img_path.suffix
            new_filename = f"{image_counter:04d}_{class_name.replace(' ', '_')}{ext}"
            new_path = eval_images_dir / new_filename
            
            # MOVE image (not copy) - this removes it from the training pool
            shutil.move(str(img_path), str(new_path))
            
            # Select caption
            if class_prompts:
                caption = class_prompts[i % len(class_prompts)]
            else:
                caption = f"An image of {class_name}"
            
            # Create label
            label = class_name.replace(' ', '_').title().replace('_', '')
            
            # Add to metadata
            metadata[new_filename] = {
                'label': label,
                'caption': caption,
                'class': class_name,
                'status': 'diseased',
                'original_path': str(img_path)
            }
            
            image_counter += 1
            diseased_count += 1
            
        except Exception as e:
            print(f"Error processing {img_path}: {e}")
            continue

print(f"Sampled and moved {diseased_count} diseased images")

Sampling diseased classes...


Diseased classes:   0%|          | 0/60 [00:00<?, ?it/s]

Sampled and moved 180 diseased images


In [7]:

print("\nSampling healthy classes...")
healthy_count = 0

for class_name in tqdm(healthy_classes, desc="Healthy classes"):
    class_dir = base_dir / class_name
    
    if not class_dir.exists():
        print(f"Warning: Directory not found for {class_name}")
        continue
    
    # Get all image files
    image_files = list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.png')) + list(class_dir.glob('*.jpeg'))
    
    if len(image_files) < 3:
        print(f"Warning: Only {len(image_files)} images available for {class_name}")
    
    # Sample 3 images (or all if less than 3)
    sampled_files = random.sample(image_files, min(3, len(image_files)))
    
    # Get prompts for this class
    class_prompts = prompts_dict.get(class_name, [])
    
    for i, img_path in enumerate(sampled_files):
        try:
            # Create unique filename
            ext = img_path.suffix
            new_filename = f"{image_counter:04d}_{class_name.replace(' ', '_')}{ext}"
            new_path = eval_images_dir / new_filename
            
            # MOVE image (not copy) - this removes it from the training pool
            shutil.move(str(img_path), str(new_path))
            
            # Select caption
            if class_prompts:
                caption = class_prompts[i % len(class_prompts)]
            else:
                caption = f"An image of healthy {class_name}"
            
            # Create label
            label = class_name.replace(' ', '_').title().replace('_', '')
            
            # Add to metadata
            metadata[new_filename] = {
                'label': label,
                'caption': caption,
                'class': class_name,
                'status': 'healthy',
                'original_path': str(img_path)
            }
            
            image_counter += 1
            healthy_count += 1
            
        except Exception as e:
            print(f"Error processing {img_path}: {e}")
            continue

print(f"Sampled and moved {healthy_count} healthy images")


Sampling healthy classes...


Healthy classes:   0%|          | 0/29 [00:00<?, ?it/s]

Sampled and moved 87 healthy images


In [8]:
# Save metadata
metadata_path = eval_base_dir / 'metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"\nSaved metadata to {metadata_path}")
print(f"Total evaluation images: {len(metadata)}")
print(f"  - Diseased: {diseased_count}")
print(f"  - Healthy: {healthy_count}")


Saved metadata to data\plantwild\evaluation\metadata.json
Total evaluation images: 267
  - Diseased: 180
  - Healthy: 87


In [10]:
# Create README
readme_content = f"""# PlantWild Evaluation Dataset

## Overview
This evaluation dataset contains {len(metadata)} images sampled from the PlantWild dataset:
- **Diseased classes**: {len(diseased_classes)} classes × 3 images = {diseased_count} images
- **Healthy classes**: {len(healthy_classes)} classes × 3 images = {healthy_count} images

## Important Notes
- **Images have been MOVED** from the main dataset to this evaluation folder
- The ingestion process will automatically skip these evaluation images
- Random seed: 42 (for reproducibility)
- These images are excluded from the image retrieval gallery

## Structure
```
evaluation/
├── images/           # {len(metadata)} image files (moved from main dataset)
├── metadata.json     # Image metadata with captions and labels
└── README.md         # This file
```

## Usage Workflow
1. **Run this sampling notebook FIRST** to extract evaluation images
2. **Then run ingestion** - it will ingest all remaining images from data/plantwild/images/
3. **Evaluate** using the images in this folder

## Metadata Format
```json
{{
  "filename.jpg": {{
    "label": "AppleBlackRot",
    "caption": "Apple black rot typically appears as...",
    "class": "apple black rot",
    "status": "diseased",
    "original_path": "path/to/original/image.jpg"
  }}
}}
```

## Class Distribution
### Diseased Classes ({len(diseased_classes)})
{chr(10).join(f'- {dc}' for dc in diseased_classes)}

### Healthy Classes ({len(healthy_classes)})
{chr(10).join(f'- {hc}' for hc in healthy_classes)}
"""

readme_path = eval_base_dir / 'README.md'
with open(readme_path, 'w', encoding='utf-8') as f:
    f.write(readme_content)

print(f"Created README at {readme_path}")

Created README at data\plantwild\evaluation\README.md


In [11]:
# Summary statistics
print("\n" + "="*60)
print("EVALUATION DATASET SUMMARY")
print("="*60)
print(f"Total images: {len(metadata)}")
print(f"Diseased images: {diseased_count} ({diseased_count/len(metadata)*100:.1f}%)")
print(f"Healthy images: {healthy_count} ({healthy_count/len(metadata)*100:.1f}%)")
print(f"\nExpected:")
print(f"  - Diseased: {len(diseased_classes)} classes × 3 = {len(diseased_classes) * 3}")
print(f"  - Healthy: {len(healthy_classes)} classes × 3 = {len(healthy_classes) * 3}")
print(f"  - Total: {(len(diseased_classes) + len(healthy_classes)) * 3}")
print(f"\nFiles saved to: {eval_base_dir}")
print("="*60)


EVALUATION DATASET SUMMARY
Total images: 267
Diseased images: 180 (67.4%)
Healthy images: 87 (32.6%)

Expected:
  - Diseased: 60 classes × 3 = 180
  - Healthy: 29 classes × 3 = 87
  - Total: 267

Files saved to: data\plantwild\evaluation

EVALUATION DATASET SUMMARY
Total images: 267
Diseased images: 180 (67.4%)
Healthy images: 87 (32.6%)

Expected:
  - Diseased: 60 classes × 3 = 180
  - Healthy: 29 classes × 3 = 87
  - Total: 267

Files saved to: data\plantwild\evaluation


In [ ]:
# Visualize some samples
import matplotlib.pyplot as plt

# Sample 8 random images (4 diseased, 4 healthy)
diseased_samples = [k for k, v in metadata.items() if v['status'] == 'diseased']
healthy_samples = [k for k, v in metadata.items() if v['status'] == 'healthy']

sample_diseased = random.sample(diseased_samples, min(4, len(diseased_samples)))
sample_healthy = random.sample(healthy_samples, min(4, len(healthy_samples)))
samples = sample_diseased + sample_healthy

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for idx, filename in enumerate(samples):
    img_path = eval_images_dir / filename
    img = Image.open(img_path)
    
    meta = metadata[filename]
    
    axes[idx].imshow(img)
    axes[idx].axis('off')
    title = f"{meta['status'].upper()}: {meta['label']}\n{meta['caption'][:50]}..."
    axes[idx].set_title(title, fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Verify the images were moved (not copied)
print("\n" + "="*60)
print("VERIFICATION: Images should be MOVED, not copied")
print("="*60)

# Count remaining images in the main dataset
remaining_count = 0
for class_name in classes:
    class_dir = base_dir / class_name
    if class_dir.exists():
        image_files = list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.png')) + list(class_dir.glob('*.jpeg'))
        remaining_count += len(image_files)

print(f"Remaining images in main dataset: {remaining_count}")
print(f"Images in evaluation folder: {len(metadata)}")
print(f"\nNext step: Run ingestion_plantwild.ipynb to ingest the {remaining_count} remaining images")
print("="*60)

## Restore Evaluation Images

Use this section to move the evaluation images back to their original locations in the main dataset.

In [1]:
# Move evaluation images back to their original locations
from pathlib import Path
import shutil
import json

# Load metadata
eval_base_dir = Path('data/plantwild/evaluation')
metadata_path = eval_base_dir / 'metadata.json'

with open(metadata_path, 'r') as f:
    metadata = json.load(f)

print(f"Found {len(metadata)} images to restore")
print("="*60)

# Move images back
restored_count = 0
failed_moves = []

for filename, meta in metadata.items():
    eval_img_path = eval_base_dir / 'images' / filename
    original_path = Path(meta['original_path'])
    
    if not eval_img_path.exists():
        print(f"Warning: Evaluation image not found: {filename}")
        failed_moves.append((filename, "Source not found"))
        continue
    
    try:
        # Create original directory if it doesn't exist
        original_path.parent.mkdir(parents=True, exist_ok=True)
        
        # Move back to original location
        shutil.move(str(eval_img_path), str(original_path))
        restored_count += 1
        
        if restored_count % 50 == 0:
            print(f"Restored {restored_count}/{len(metadata)} images...")
        
    except Exception as e:
        print(f"Error restoring {filename}: {e}")
        failed_moves.append((filename, str(e)))

print("="*60)
print(f"Successfully restored: {restored_count}/{len(metadata)} images")

if failed_moves:
    print(f"\nFailed to restore {len(failed_moves)} images:")
    for filename, error in failed_moves[:10]:  # Show first 10 errors
        print(f"  - {filename}: {error}")
else:
    print("\nAll images successfully restored!")
    print("\nYou can now delete the evaluation folder if desired:")
    print(f"  - {eval_base_dir / 'images'}")
    print(f"  - {metadata_path}")

Found 209 images to restore
Restored 50/209 images...Restored 50/209 images...

Restored 100/209 images...
Restored 150/209 images...
Restored 100/209 images...
Restored 150/209 images...
Restored 200/209 images...
Successfully restored: 209/209 images

All images successfully restored!

You can now delete the evaluation folder if desired:
  - data\plantwild\evaluation\images
  - data\plantwild\evaluation\metadata.json
Restored 200/209 images...
Successfully restored: 209/209 images

All images successfully restored!

You can now delete the evaluation folder if desired:
  - data\plantwild\evaluation\images
  - data\plantwild\evaluation\metadata.json
